# Exp 2 — Q(d) analysis (v2)

Fits Q(d) per corpus and does the checks that decide H1 vs H2:

- **β with document-bootstrap 95% CI** (full range).
- **Model comparison**: power-law vs exponential vs curved (log-normal-ish) on
  log Q, by AIC — this is what licenses the word "power law".
- **Restricted-range β** (d≥8, d≥16): drops the smallest token bands where a
  token isn't a comparable linguistic unit (the only place tokenization can bias
  the *slope*). If a shallow β climbs here, the near bins were the problem.
- **Robustness**: β in nats, null-control β (near+far shuffle), α(P) overlap,
  and n_pos bins per corpus. Small-n corpora flagged.

Note: a log–log slope is invariant to constant rescaling of either axis, so
token→char/word normalization shifts intercepts, **not** β. Cross-lingual β
comparison is therefore already tokenization-robust; no distance normalization
is applied here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, math
import numpy as np

BASE = Path('/content/drive/MyDrive/LRTIA/Results/exp2_increment_shuffle/llama')
MAIN = Path('/content/drive/MyDrive/LRTIA/Results/corpus_expansion_longrange/llama')  # alpha(P)
SMALL_N = 20   # flag corpora with fewer docs

def q_bins(records, ppl_key='B_ppl_mean', a_key='A_ppl'):
    """Per-interval (distance, mean Q, n) aggregated across targets."""
    bi={}
    for r in records:
        for p in r['pairs']:
            b=p.get(ppl_key); a=p.get(a_key)
            if b is None or a is None or (isinstance(b,float) and math.isnan(b)): continue
            if p['width']<2: continue  # drop degenerate (1,2) no-op band (fp16-noise leverage point)
            bi.setdefault(p['i'],{'d':p['distance'],'qs':[]})['qs'].append((b-a)/p['width'])
    return [(v['d'], sum(v['qs'])/len(v['qs']), len(v['qs'])) for i,v in sorted(bi.items())]

def _ols(X, y):
    c,_,_,_=np.linalg.lstsq(X, y, rcond=None)
    rss=float(((y-X@c)**2).sum())
    return c, rss

def fit_powerlaw(bins, dmin=0.0):
    pts=[(d,q) for (d,q,_) in bins if q>0 and d>=dmin]
    if len(pts)<4: return None
    d=np.array([p[0] for p in pts]); q=np.array([p[1] for p in pts])
    x=np.log(d); y=np.log(q); n=len(x)
    (c_pl,rss_pl)=_ols(np.vstack([np.ones(n),x]).T, y)
    beta=-c_pl[1]
    tss=float(((y-y.mean())**2).sum()); r2=1-rss_pl/tss if tss>0 else float('nan')
    return {'beta':beta,'n':n,'r2':r2,'rss':rss_pl,'x':x,'y':y,'d':d}

def model_compare(bins):
    """AIC: power-law (y~logd) vs exponential (y~d) vs curved (y~logd+logd^2)."""
    f=fit_powerlaw(bins)
    if f is None: return None
    x=f['x']; y=f['y']; d=f['d']; n=f['n']
    def aic(rss,k): return n*math.log(rss/n)+2*k if rss>0 else -math.inf
    _,rss_pl=_ols(np.vstack([np.ones(n),x]).T, y)          # power law
    _,rss_ex=_ols(np.vstack([np.ones(n),d]).T, y)          # exponential
    _,rss_cu=_ols(np.vstack([np.ones(n),x,x**2]).T, y)     # curved
    A={'power':aic(rss_pl,2),'exp':aic(rss_ex,2),'curved':aic(rss_cu,3)}
    best='power' if A['power']-min(A.values())<=2 else min(A,key=A.get)  # parsimony: power unless dAIC>2
    return {'best':best,'dAIC_exp_minus_pl':A['exp']-A['power'],
            'dAIC_curved_minus_pl':A['curved']-A['power']}

def bootstrap_beta(records, n_boot=1000, seed=12345, dmin=0.0):
    if len(records)<3: return None
    rng=np.random.default_rng(seed); idx=np.arange(len(records)); bs=[]
    for _ in range(n_boot):
        samp=[records[j] for j in rng.choice(idx,size=len(records),replace=True)]
        f=fit_powerlaw(q_bins(samp),dmin=dmin)
        if f: bs.append(f['beta'])
    if len(bs)<n_boot*0.5: return None
    a=np.array(bs); return float(np.percentile(a,2.5)), float(np.percentile(a,97.5))

def alpha_from_main(cache, dmin=2.0):
    if not cache.exists(): return None
    recs=json.load(open(cache)); bi={}
    for r in recs:
        cs,op,sp=r['context_lengths'],r['ordered_ppl'],r['shuffled_ppl']
        for i in range(1,len(cs)):
            if cs[i-1]==0: continue
            w=cs[i]-cs[i-1]
            gr=((op[i-1]-op[i])-(sp[i-1]-sp[i]))/w
            bi.setdefault(i,{'d':math.sqrt(cs[i-1]*cs[i]),'g':[]})['g'].append(gr)
    bins=[(v['d'],sum(v['g'])/len(v['g']),0) for v in bi.values()]
    f=fit_powerlaw(bins,dmin=dmin)   # dmin=2 matches Q's range (degenerate near band dropped)
    return f['beta'] if f else None

def f2(x,p=3): return f"{x:.{p}f}" if isinstance(x,(int,float)) and not (isinstance(x,float) and math.isnan(x)) else "--"

FIT={}
rows=[]
for cache in sorted(BASE.glob('*.json')):
    recs=json.load(open(cache))
    if not recs: continue
    bins=q_bins(recs)
    f=fit_powerlaw(bins); f8=fit_powerlaw(bins,8); f16=fit_powerlaw(bins,16)
    ci=bootstrap_beta(recs)
    mc=model_compare(bins)
    ac=MAIN/cache.name
    a=alpha_from_main(ac,2.0); a8=alpha_from_main(ac,8.0); a16=alpha_from_main(ac,16.0)
    has_ctrl=any('null_full_ppl_mean' in p for r in recs for p in r['pairs'])
    nb=fit_powerlaw(q_bins(recs,ppl_key='null_full_ppl_mean')) if has_ctrl else None
    fn=fit_powerlaw(q_bins(recs,ppl_key='B_nll_mean',a_key='A_nll'))
    FIT[cache.stem]=(recs,f)
    rows.append(dict(corpus=cache.stem, n=len(recs), npos=(f['n'] if f else 0),
        beta=(f['beta'] if f else None), r2=(f['r2'] if f else None),
        ci=ci, best=(mc['best'] if mc else None),
        dexp=(mc['dAIC_exp_minus_pl'] if mc else None),
        b8=(f8['beta'] if f8 else None), b16=(f16['beta'] if f16 else None),
        bn=(fn['beta'] if fn else None), nbeta=(nb['beta'] if nb else None),
        alpha=a, a8=a8, a16=a16))

print("=== HEADLINE: power-law fit, full range ===")
print(f"{'corpus':<24}{'n':>4}{'npos':>5}{'beta':>8}{'95% CI':>16}{'r2':>7}{'best':>8}{'alpha(P)':>10}")
print('-'*82)
for r in rows:
    flag=' *' if r['n']<SMALL_N else ''
    ci=f"[{r['ci'][0]:.2f},{r['ci'][1]:.2f}]" if r['ci'] else '--'
    print(f"{r['corpus']+flag:<24}{r['n']:>4}{r['npos']:>5}{f2(r['beta']):>8}{ci:>16}{f2(r['r2']):>7}{str(r['best']):>8}{f2(r['alpha']):>10}")

print("\n=== BETA vs ALPHA, range-matched  (same distance floor; a = P(d) exponent) ===")
print(f"{'corpus':<22}{'b(full)':>8}{'a(full)':>8}{'b(d>=8)':>9}{'a(d>=8)':>9}{'b(d>=16)':>10}{'a(d>=16)':>10}")
print('-'*76)
for r in rows:
    print(f"{r['corpus']:<22}{f2(r['beta']):>8}{f2(r['alpha']):>8}{f2(r['b8']):>9}{f2(r['a8']):>9}{f2(r['b16']):>10}{f2(r['a16']):>10}")
print("\n=== ROBUSTNESS  (beta_nats, null-control beta, dAIC = exp - power) ===")
print(f"{'corpus':<24}{'beta':>8}{'beta_nats':>10}{'null_beta':>10}{'dAIC':>8}")
print('-'*60)
for r in rows:
    print(f"{r['corpus']:<24}{f2(r['beta']):>8}{f2(r['bn']):>10}{f2(r['nbeta']):>10}{f2(r['dexp'],1):>8}")
print("\n* = n <", SMALL_N, "docs (underpowered).  b~a & best=power => strong H1; b>a => partial dissociation.")

In [ ]:
# Log-log Q(d) overlay.
import matplotlib.pyplot as plt
plt.figure(figsize=(7.5,5.5))
for name,(recs,f) in FIT.items():
    bins=q_bins(recs)
    ds=[d for d,g,_ in bins if g>0]; gs=[g for d,g,_ in bins if g>0]
    if ds:
        lab=f"{name}  b={f['beta']:.2f}" if f else name
        plt.loglog(ds,gs,marker='o',ms=4,label=lab)
plt.xlabel('distance d  (geometric midpoint, tokens)'); plt.ylabel('Q(d)   [ppl per token]')
plt.title('Exp 2 - span-internal order contribution Q(d)')
plt.legend(fontsize=7,ncol=2); plt.grid(True,which='both',alpha=0.3)
out='/content/drive/MyDrive/LRTIA/Results/exp2_increment_shuffle/Qd_loglog.png'
plt.savefig(out,dpi=150,bbox_inches='tight'); print('saved',out); plt.show()